In [2]:
import json
import pandas as pd
import numpy as np
from scipy import stats

In [3]:
# load the data from provided results.json file
with open("results.json", 'r') as f:
    data = json.load(f)

rows = []
# go through all the data 
for entry in data:
    
    # if there are no scores then we ignore it
    if entry["scores"] is None:
        continue

    # create row for every relevant part of the data
    rows.append({
        "output_id":      entry["output_id"],
        "transcript_num": entry["transcript_num"],
        "seed":           entry["seed"],
        "faithfulness":   entry["scores"]["faithfulness"]["score"],
        "construct_correctness": entry["scores"]["construct_correctness"]["score"],
        "coverage":       entry["scores"]["coverage"]["score"],
        "label_quality":  entry["scores"]["label_quality"]["score"],
    })

# create a dataframe 
df = pd.DataFrame(rows)

In [4]:
# mapping for the cleaned output to model and prompt type
mapping = {
    "output_1" : {"model" : "gemma", "prompt" : "cot"},
    "output_2" : {"model" : "mistral", "prompt" : "cot"},
    "output_3" : {"model" : "qwen", "prompt" : "cot"},
    "output_4" : {"model" : "gemma", "prompt" : "few_shot"},
    "output_5" : {"model" : "mistral", "prompt" : "few_shot"},
    "output_6" : {"model" : "qwen", "prompt" : "few_shot"},
    "output_7" : {"model" : "gemma", "prompt" : "zero_shot"},
    "output_8" : {"model" : "mistral", "prompt" : "zero_shot"},
    "output_9" : {"model" : "qwen", "prompt" : "zero_shot"},
    "output_10" : {"model" : "gemma", "prompt" : "cot"},
    "output_11" : {"model" : "mistral", "prompt" : "cot"},
    "output_12" : {"model" : "qwen", "prompt" : "cot"},
    "output_13" : {"model" : "gemma", "prompt" : "few_shot"},
    "output_14" : {"model" : "mistral", "prompt" : "few_shot"},
    "output_15" : {"model" : "qwen", "prompt" : "few_shot"},
    "output_16" : {"model" : "gemma", "prompt" : "zero_shot"},
    "output_17" : {"model" : "mistral", "prompt" : "zero_shot"},
    "output_18" : {"model" : "qwen", "prompt" : "zero_shot"},
    "output_19" : {"model" : "gemma", "prompt" : "cot"},
    "output_20" : {"model" : "mistral", "prompt" : "cot"},
    "output_21" : {"model" : "qwen", "prompt" : "cot"},
    "output_22" : {"model" : "gemma", "prompt" : "few_shot"},
    "output_23" : {"model" : "mistral", "prompt" : "few_shot"},
    "output_24" : {"model" : "qwen", "prompt" : "few_shot"},
    "output_25" : {"model" : "gemma", "prompt" : "zero_shot"},
    "output_26" : {"model" : "mistral", "prompt" : "zero_shot"},
    "output_27" : {"model" : "qwen", "prompt" : "zero_shot"},
    "output_28" : {"model" : "gemma", "prompt" : "cot"},
    "output_29" : {"model" : "mistral", "prompt" : "cot"},
    "output_30" : {"model" : "qwen", "prompt" : "cot"},
    "output_31" : {"model" : "gemma", "prompt" : "few_shot"},
    "output_32" : {"model" : "mistral", "prompt" : "few_shot"},
    "output_33" : {"model" : "qwen", "prompt" : "few_shot"},
    "output_34" : {"model" : "gemma", "prompt" : "zero_shot"},
    "output_35" : {"model" : "mistral", "prompt" : "zero_shot"},
    "output_36" : {"model" : "qwen", "prompt" : "zero_shot"},
    "output_37" : {"model" : "gemma", "prompt" : "cot"},
    "output_38" : {"model" : "mistral", "prompt" : "cot"},
    "output_39" : {"model" : "qwen", "prompt" : "cot"},
    "output_40" : {"model" : "gemma", "prompt" : "few_shot"},
    "output_41" : {"model" : "mistral", "prompt" : "few_shot"},
    "output_42" : {"model" : "qwen", "prompt" : "few_shot"},
    "output_43" : {"model" : "gemma", "prompt" : "zero_shot"},
    "output_44" : {"model" : "mistral", "prompt" : "zero_shot"},
    "output_45" : {"model" : "qwen", "prompt" : "zero_shot"},
    "output_46" : {"model" : "gemma", "prompt" : "cot"},
    "output_47" : {"model" : "mistral", "prompt" : "cot"},
    "output_48" : {"model" : "qwen", "prompt" : "cot"},
    "output_49" : {"model" : "gemma", "prompt" : "few_shot"},
    "output_50" : {"model" : "mistral", "prompt" : "few_shot"},
    "output_51" : {"model" : "qwen", "prompt" : "few_shot"},
    "output_52" : {"model" : "gemma", "prompt" : "zero_shot"},
    "output_53" : {"model" : "mistral", "prompt" : "zero_shot"},
    "output_54" : {"model" : "qwen", "prompt" : "zero_shot"},

}

In [5]:
# help function to get model and prompt 
def get_model_prompt(output_id):
    return mapping[output_id]["model"], mapping[output_id]["prompt"]

df["model"] = df["output_id"].apply(lambda x : get_model_prompt(x)[0])
df["prompt"] = df["output_id"].apply(lambda x : get_model_prompt(x)[1])
df["combination"] = df["model"] + "_" + df["prompt"]


In [6]:
# criteria that we test on in LLM-as-a-judge
criteria = ["faithfulness", "construct_correctness", "coverage", "label_quality"]   


In [10]:
# calculate per model metrics
per_model_means = df.groupby("model")[criteria].mean()

per_model_means["overall"] = per_model_means.mean(axis=1).round(3)
per_model_means = per_model_means.sort_values("overall", ascending=False)
per_model_means

,faithfulness,construct_correctness,coverage,label_quality,overall
model,,,,,
mistral,4.672222,4.700000,4.416667,4.688889,4.619
gemma,4.666667,4.738889,4.322222,4.733333,4.615
qwen,4.644444,4.677778,4.350000,4.700000,4.593


In [18]:
# calculate per prompt metrics
per_prompt_means = df.groupby("prompt")[criteria].mean()

per_prompt_means["overall"] = per_prompt_means.mean(axis=1).round(3)
per_prompt_means = per_prompt_means.sort_values("overall", ascending=False)
per_prompt_means

,faithfulness,construct_correctness,coverage,label_quality,overall
prompt,,,,,
cot,4.794444,4.705556,4.444444,4.761111,4.676
zero_shot,4.627778,4.716667,4.322222,4.705556,4.593
few_shot,4.561111,4.694444,4.322222,4.655556,4.558


In [19]:
per_model_means["variance"] = per_model_means[criteria].var(axis=1).round(3)
per_model_means

,faithfulness,construct_correctness,coverage,label_quality,overall,variance
model,,,,,,
mistral,4.672222,4.700000,4.416667,4.688889,4.619,0.018
gemma,4.666667,4.738889,4.322222,4.733333,4.615,0.039
qwen,4.644444,4.677778,4.350000,4.700000,4.593,0.027


In [16]:
per_prompt_means["variance"] = per_prompt_means[criteria].var(axis=1).round(3)
per_prompt_means

,faithfulness,construct_correctness,coverage,label_quality,overall,variance
prompt,,,,,,
cot,4.794444,4.705556,4.444444,4.761111,4.676,0.025
zero_shot,4.627778,4.716667,4.322222,4.705556,4.593,0.034
few_shot,4.561111,4.694444,4.322222,4.655556,4.558,0.028


In [38]:
combination = df.groupby('combination')[criteria].mean()
combination["overall"] = combination.mean(axis = 1).round(3)
combination = combination.sort_values("overall", ascending=False)
combination

,faithfulness,construct_correctness,coverage,label_quality,overall
combination,,,,,
gemma_cot,4.783333,4.683333,4.500000,4.833333,4.700
qwen_cot,4.866667,4.733333,4.333333,4.750000,4.671
mistral_cot,4.733333,4.700000,4.500000,4.700000,4.658
mistral_zero_shot,4.700000,4.650000,4.483333,4.716667,4.638
gemma_zero_shot,4.716667,4.783333,4.266667,4.733333,4.625
qwen_few_shot,4.600000,4.583333,4.500000,4.683333,4.592
mistral_few_shot,4.583333,4.750000,4.266667,4.650000,4.562
gemma_few_shot,4.500000,4.750000,4.200000,4.633333,4.521
qwen_zero_shot,4.466667,4.716667,4.216667,4.666667,4.517
